## Setup

In [1]:
from evosax import Strategies, ParameterReshaper, EvoState

In [2]:
from jaxcmr.memorysearch import FittingStrategy, BaseCMR, InstanceCMR, create_predict_fn, predict_trials, predict_trial, experience, start_retrieving
import pytest
import jax
import json
from jax import numpy as jnp, vmap, tree_util, random
from jax.nn import sigmoid
from jaxcmr.datasets import load_data, generate_trial_mask
from jaxcmr.helpers import latin_hypercube_sampling, scale_to_bounds

In [3]:
parameter_path = "D:/data/base_cmr_parameters.json"
data_path = "D:/data/{}.h5"

model_create_fn = BaseCMR
data_tag = "HealyKahana2014"
trial_query = "data['listtype'] == -1"
rng = jax.random.PRNGKey(0)

with open(parameter_path, "r") as f:
    parameters = json.load(f)

data = load_data(data_path.format(data_tag))
trial_mask = generate_trial_mask(data, trial_query)
subject_trials = jnp.array(data["recalls"][trial_mask].reshape(126, 28, 16))
subject_presentations = jnp.array(data["pres_itemnos"][trial_mask].reshape(126, 28, 16))

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


## Base Example

In [4]:
# def scale_params(params: dict, bounds: dict[str, list[float]]):
#     return tree_util.tree_map(lambda x, y: y[0] + x * (y[1] - y[0]), params, bounds)

def scale_params(params: dict, bounds: dict[str, list[float]]):
    return jax.tree_util.tree_map(lambda x, y: y[0] + sigmoid(x) * (y[1] - y[0]), params, bounds)

reshape_params = ParameterReshaper(
    {key: parameters["fixed"][key] for key in parameters["free"]}).reshape

ParameterReshaper: 12 parameters detected for optimization.


In [15]:
# for s_name in ["SimpleES", "SimpleGA", "PSO", "DE", "Sep_CMA_ES",
#                "Full_iAMaLGaM", "Indep_iAMaLGaM", "MA_ES", "LM_MA_ES",
#                "RmES", "GLD", "SimAnneal", "GESMR_GA", "SAMR_GA"]:

subject_index = 11
popsize = 50

assert subject_index > 0

for s_name in ["DE"]:
    strategy = Strategies[s_name](popsize=popsize, num_dims=len(parameters["free"]))
    es_params = strategy.default_params
    state = strategy.initialize(rng, es_params)

    single_subject_objective_fn = vmap(
        create_predict_fn(
            model_create_fn,
            subject_presentations[subject_index-1],
            subject_trials[subject_index-1],
        )
    )

    for t in range(1000):
        rng, rng_iter = random.split(rng)

        if t == 0:
            x = scale_to_bounds(
                latin_hypercube_sampling(rng_iter, len(parameters["free"]), popsize), -7, 7)
            # x = latin_hypercube_sampling(rng_iter, len(parameters["free"]), 15)
            state = state.replace(archive=x)
        else:
            x, state = strategy.ask(rng_iter, state, es_params)
4

DE - # Gen: 50|Fitness: 692.77|Params: [ -5.9714665 -29.591017    8.650184  -14.084873   -0.6522512  -2.5469475
 -41.51385     6.4639916 -20.152964    7.042699   -8.843367   -3.446415 ]
DE - # Gen: 100|Fitness: 666.57|Params: [  -6.0331917   15.327321    11.774626   -49.041943    -1.5532244
 -111.933205    29.112158     5.5859528  -17.684708    -5.1759944
   -3.4517398   -6.01666  ]
DE - # Gen: 150|Fitness: 651.49|Params: [  -4.3939104   14.483908     2.564713  -205.58565     -0.480975
 -316.04105   -202.57916      1.247547    -9.100348    -6.5277433
   -2.9759035   -8.304116 ]
DE - # Gen: 200|Fitness: 649.09|Params: [-3.6533005e+00  7.0506659e+00  1.7199513e+00 -3.3523901e+02
 -3.5224658e-02 -1.0210113e+03 -1.1808274e+02  1.1453044e+00
 -7.1759353e+00 -3.0247865e+00 -3.0914195e+00 -7.8726850e+00]
DE - # Gen: 250|Fitness: 647.96|Params: [-2.6981533e+00  9.2088175e+00  1.2143596e+00 -2.6162646e+02
  7.7911156e-01 -3.4317166e+03 -6.8000412e+00  1.2457602e+00
 -5.6862068e+00 -4.4101958e+0

## With Latin Hypercube Initialization

In [20]:
from scipy.stats import qmc
sampler = qmc.LatinHypercube(d=len(parameters["free"]))
sample = sampler.random(n=15)
l_bounds = [parameters["free"][key][0] for key in parameters["free"]]
u_bounds = [parameters["free"][key][1] for key in parameters["free"]]
sample_scaled = qmc.scale(sample, l_bounds, u_bounds)
sample_scaled.shape

(15, 12)

In [21]:
subject_index = 1

for s_name in ["DE"]:
    strategy = Strategies[s_name](popsize=15, num_dims=len(parameters["free"]))
    es_params = strategy.default_params
    state = strategy.initialize(rng, es_params)

    single_subject_objective_fn = vmap(
        create_predict_fn(
            model_create_fn,
            subject_presentations[subject_index],
            subject_trials[subject_index],
        )
    )

    for t in range(1000):
        rng, rng_iter = random.split(rng)
        x, state = strategy.ask(rng_iter, state, es_params)

        x_dict = scale_params(reshape_params(x), parameters["free"])
        fitness = single_subject_objective_fn(x_dict)
        state = strategy.tell(x, fitness, state, es_params)

        if (t + 1) % 50 == 0:
            print(
                "{} - # Gen: {}|Fitness: {:.2f}|Params: {}".format(
                    s_name, t + 1, state.best_fitness, state.best_member
                )
            )
    print(20 * "=")

DE - # Gen: 50|Fitness: 710.08|Params: [ -2.6030276    0.4507178    0.14102447   0.21865317   8.7598915
   0.72786325  -8.239449   -36.25319     -2.5907693    1.6578391
  -3.8501668   -5.0247493 ]
DE - # Gen: 100|Fitness: 707.98|Params: [-2.7340262e+00  5.4749730e-03 -4.4978593e-02  2.6491126e-01
  7.8548145e+00  2.3471320e+00 -1.4746104e+01 -2.0450079e+01
 -3.5333195e+00  4.2172709e+00 -3.6830435e+00 -5.4795446e+00]
DE - # Gen: 150|Fitness: 707.58|Params: [ -2.8159      -3.110306    -0.06954081   0.52926284   7.1330543
 -25.890135   -33.15537    -55.62908     -3.8196702    5.246639
  -3.5413175   -5.8979096 ]
DE - # Gen: 200|Fitness: 707.25|Params: [-2.87643194e+00 -1.54739499e+00 -1.59711987e-01  1.17530048e+00
  3.61126375e+00 -2.10794507e+03 -2.65266228e+01 -1.06366875e+02
 -5.06645918e+00  5.98077583e+00 -3.58297825e+00 -5.75906801e+00]
DE - # Gen: 250|Fitness: 707.03|Params: [-3.2284260e+00 -2.6446596e+01  3.7469636e-04  5.0642180e-01
  2.3140135e+00 -1.0418591e+04 -3.8359918e+02

I just have to give it my custom x and the corresponding fitness values along with es_params and state. 

So what will the strategy be, concretely?
Init values will be created outside jax, like everything else.
For iteration 0, use init_x for the ask tell loop.
Great.